# M10 Part A.5 — run MAMMA on BEHAVE → SMPL-X exports

**Separate env from the render notebook.** MAMMA needs its own conda env (CUDA 12.4 + from-source CUDA kernels), which conflicts with the fork's Colab build — so this notebook ONLY produces per-frame SMPL-X exports to Drive. `M10 - smpl_person_render` §6 then consumes them with `--smpl_source mamma`.

Pipeline: BEHAVE 4 views → (adapter) MAMMA footage+calib → `inference run` → (adapter) per-frame `.npz` exports.

> ⚠️ **This is the heavy/fragile part.** The install is a conda env pinned to CUDA 12.4 that COMPILES detectron2 + pytorch_sdf, plus several GB of weights — two of them (MAMMA weights, SMPL-X) are **license-gated and need YOUR sign-in**. Expect to debug §3–§4. Each stage is its own cell.
> 
> **VERIFY on first run (from MAMMA's `ma_vis` overlays):** the body must sit ON the person. If cameras look mirrored/behind → re-run §6 with `--extrinsics cam2world`; if the body is wildly rotated → try `--quat_order xyzw`. (BEHAVE `local2world`=cam2world; MAMMA's calib convention + quat order are unconfirmed.)

## 1. GPU + mount + clone (fork adapter, behave-dataset reader, mamma)

In [ ]:
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
import os, sys
os.chdir('/content')
for repo, url in [('gaussian-surfel-local-map','https://github.com/steptang/gaussian-surfel-local-map.git'),
                  ('behave-dataset','https://github.com/xiexh20/behave-dataset.git'),
                  ('mamma','https://github.com/cuevhv/mamma.git')]:
    if not os.path.isdir(f'/content/{repo}'):
        !git clone -q {url}
!cd /content/gaussian-surfel-local-map && git pull -q
sys.path.insert(0, '/content/gaussian-surfel-local-map')
sys.path.insert(0, '/content/behave-dataset')   # data.kinect_transform.KinectTransform
!pip -q install pyyaml opencv-python-headless
print('repos ready')

## 2. Paths

In [ ]:
DRIVE     = '/content/drive/MyDrive/Research'
SEQ       = 'Date03_Sub05_boxtiny'
SEQ_DIR   = f'{DRIVE}/datasets/behave/sequences/{SEQ}'      # raw BEHAVE (t*.000)
N_SELECT  = 24                                              # MUST match the converter (so frame i == timestep_i)
MAMMA_IN  = '/content/mamma_in'                             # local footage+calib (fast)
EXPORTS   = f'{DRIVE}/datasets/mamma_exports/{SEQ}'         # per-frame .npz -> render notebook reads this
import os; assert os.path.isdir(SEQ_DIR), SEQ_DIR; print('ok')

## 3. Install MAMMA (conda env, CUDA 12.4, compiles kernels) — SLOW / may need debugging
Follows `mamma/docs/INSTALL.md`. Colab has no conda by default → install micromamba first. If the CUDA-12.4 pin or the detectron2/pytorch_sdf compile fights Colab, that's the expected failure point (INSTALL.md notes the 12.4 pin is critical).

In [ ]:
%cd /content/mamma
# micromamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
MM = '/content/mamma/bin/micromamba'; ROOT = '/content/mm'
!$MM create -y -r $ROOT -f requirements/mamma_conda.yaml
# CUDA 12.4 toolkit must be on PATH before the compiling pip layer:
import os
os.environ['CUDA_HOME'] = f'{ROOT}/envs/mamma'   # conda env ships cuda-12.4 (per INSTALL.md); adjust if elsewhere
!$MM run -r $ROOT -n mamma bash -lc 'export CUDA_HOME=$CONDA_PREFIX PATH=$CONDA_PREFIX/bin:$PATH; \
  pip install -r requirements/requirements.txt && \
  pip install --no-build-isolation -r requirements/requirements_no_build_isolation.txt'
print('env built (verify no compile errors above)')

## 4. Weights — public wgets + gated (LICENSE sign-in required, do these yourself)
MAMMA weights + SMPL-X are gated: accept the licenses (MAMMA repo / smpl-x.is.tue.mpg.de) and run the repo's download scripts (they prompt for credentials), or download manually into the paths below.

In [ ]:
%cd /content/mamma
!mkdir -p data/weights/yolo data/weights/sam2
!wget -q -O data/weights/yolo/yolo12x.pt https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo12x.pt
!wget -q -O data/weights/sam2/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
# GATED — run and follow the sign-in prompts (needs your accepted licenses):
MM='/content/mamma/bin/micromamba'; ROOT='/content/mm'
# !$MM run -r $ROOT -n mamma bash data/download_mamma_weights.sh --all
# !$MM run -r $ROOT -n mamma bash data/download_smplx_locked_head.sh
print('public weights fetched; run the two gated scripts above (uncomment) after accepting licenses')

In [ ]:
# sanity: env vars + weight paths
MM='/content/mamma/bin/micromamba'; ROOT='/content/mm'
!$MM run -r $ROOT -n mamma python -m inference doctor

## 5. Build MAMMA input from BEHAVE (adapter `input`)

In [ ]:
# runs in base python (cv2 + pyyaml + KinectTransform). Writes images/ + <SEQ>.calib.yaml + <SEQ>.capture.json
!python -m tracking.behave_to_mamma input \
  --seq_dir "$SEQ_DIR" --out_root "$MAMMA_IN" --seq_name "$SEQ" --n_select $N_SELECT \
  --extrinsics world2cam --quat_order wxyz

## 6. Run MAMMA inference (in its env)

In [ ]:
%cd /content/mamma
MM='/content/mamma/bin/micromamba'; ROOT='/content/mm'
CALIB=f'{MAMMA_IN}/{SEQ}.calib.yaml'
!$MM run -r $ROOT -n mamma python -m inference run \
  --cfg configs/examples/presets/quick.yaml \
  --footage "$MAMMA_IN" --seq_name "$SEQ" --calib "$CALIB" --out-tag behave -v
# inspect ma_vis overlays under output/ before trusting ma_3d (VERIFY body-on-person; else adjust §5 flags)

## 7. Convert MAMMA ma_3d SMPL-X → per-frame exports on Drive (adapter `output`)

In [ ]:
import glob
# find MAMMA's ma_3d output dir for this run (structure varies — inspect if empty):
cands = glob.glob('/content/mamma/output/**/behave/**', recursive=True)
print('ma_3d output candidates:'); [print(' ', c) for c in cands[:20]]
MAMMA_OUT = f'/content/mamma/output/ma_3d/behave/{SEQ}'   # TODO adjust to the real path printed above
!python -m tracking.behave_to_mamma output --mamma_out_dir "$MAMMA_OUT" --exports_root "$EXPORTS"
print('exports ->', EXPORTS)

## 8. Next — render with MAMMA bodies (in the render notebook)
In **`M10 - smpl_person_render`** §6, set and run:
```python
MAMMA_ROOT     = f'{DRIVE}/datasets/mamma_exports/Date03_Sub05_boxtiny'   # = EXPORTS here
SMPL_MODEL_DIR = '/content/mamma/data/body_models/smplx_locked_head'      # SMPL-X to pose MAMMA params
# !python -m tracking.smpl_person --conv_root $CONV_ROOT --smpl_root $MAMMA_ROOT \
#   --smpl_model_dir $SMPL_MODEL_DIR --smpl_source mamma --n_select 24 --out /content/out_partA5_mamma ...
```
Then compare `person_psnr_mean`: **GT (Part A) vs MAMMA (A.5) vs the deform-rig baseline**.